In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import warnings

# Reading rows bins the video timestamps, which this guide does not use.
warnings.filterwarnings("ignore", message=".*binned automatically")

In [3]:
from collections.abc import Iterator
from typing import cast

import numpy as np
import polars as pl
from IPython.display import display
from maite_datasets.multiobject_tracking import (
    MultiobjectTrackingTargetTuple,
    SingleFrameObjectTrackingTargetTuple,
    VideoFrameTuple,
)

from dataeval import Metadata
from dataeval.config import use_seed
from dataeval.protocols import DatasetMetadata, DatumMetadata, MultiobjectTrackingDatum, VideoFrame
from dataeval.quality import Outliers
from dataeval.types import SourceIndex

pl.Config.set_tbl_width_chars(160)

polars.config.Config

In [4]:
N_VIDEOS, N_FRAMES, HEIGHT, WIDTH = 8, 200, 72, 96
BUS_VIDEO, BUS_FRAMES = 3, 12
LATE_VIDEO, LATE_FRAME = 7, 120
VEHICLES = {0: "car", 1: "car", 2: "truck", 3: "bus"}


def track_ids(video: int, frame: int) -> list[int]:
    """The tracks present in one frame."""
    return [0, 1, 2, 3] if video == BUS_VIDEO and frame < BUS_FRAMES else [0, 1, 2]


def frame_targets(video: int) -> list[SingleFrameObjectTrackingTargetTuple]:
    """One target per frame, holding a box for each track present."""
    targets = []
    for frame in range(N_FRAMES):
        ids = track_ids(video, frame)
        targets.append(
            SingleFrameObjectTrackingTargetTuple(
                boxes=np.tile(np.array([10, 10, 22, 22], dtype=np.float32), (len(ids), 1)),
                labels=np.zeros(len(ids), dtype=np.int64),
                scores=np.ones(len(ids), dtype=np.float32),
                track_ids=np.array(ids, dtype=np.int64),
            )
        )
    return targets


class BlankVideo:
    """A stand-in for a decoded video: the right number of frames, all zeros, one of them stamped 3 hours late."""

    def __init__(self, video: int) -> None:
        self.video = video

    def __iter__(self) -> Iterator[VideoFrame]:
        blank = np.zeros((3, HEIGHT, WIDTH), dtype=np.uint8)
        for index in range(N_FRAMES):
            late = 3 * 3600 if (self.video, index) == (LATE_VIDEO, LATE_FRAME) else 0
            yield VideoFrameTuple(pixels=blank, time_s=index / 30 + late, pts=index, frame_index=index)


class TrackingDataset:
    """A minimal MAITE multi-object tracking dataset."""

    def __init__(self) -> None:
        self.metadata = DatasetMetadata({"id": "conditional-outliers", "index2label": {0: "vehicle"}})
        self._data: list[MultiobjectTrackingDatum] = [
            (
                BlankVideo(video),
                MultiobjectTrackingTargetTuple(frame_tracks=frame_targets(video)),
                cast(DatumMetadata, {"id": f"video_{video}", "height": HEIGHT, "width": WIDTH}),
            )
            for video in range(N_VIDEOS)
        ]

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, index: int) -> MultiobjectTrackingDatum:
        return self._data[index]


metadata = Metadata(TrackingDataset())
print(metadata.level_counts)

{'sequence': 8, 'unit': 1600, 'track': 25, 'instance': 4812}


In [5]:
rng = np.random.default_rng(0)

video = np.repeat(np.arange(N_VIDEOS), N_FRAMES)
frame = np.tile(np.arange(N_FRAMES), N_VIDEOS)
weather = np.where(video == 5, "snow", "clear")
weather[(video == 0) & (frame == 80)] = "snow"
hour = video * 3.0 + frame * 0.015
brightness = 120 + 80 * np.cos((hour - 13) / 24 * 2 * np.pi) + rng.normal(0, 4, hour.size)
brightness[(video == 0) & (frame == 140)] = 195.0
exposure = 8.0 + video + np.random.default_rng(1).normal(0, 0.1, video.size)
exposure[(video == 4) & (frame == 50)] = 9.0
metadata.add_factors({"weather": weather, "hour": hour, "brightness": brightness, "exposure": exposure}, level="unit")

detections = [(v, f, t) for v in range(N_VIDEOS) for f in range(N_FRAMES) for t in track_ids(v, f)]
d_video, d_frame, d_track = (np.array(column) for column in zip(*detections, strict=True))
vehicle = np.array([VEHICLES[t] for t in d_track])
box_y = rng.uniform(0.2, 0.9, d_track.size)
box_size = np.where(vehicle == "car", 1.0, 2.0) * (0.02 + 0.08 * box_y) + rng.normal(0, 0.004, d_track.size)

large_car = np.flatnonzero((d_video == 2) & (d_frame == 100) & (d_track == 0))[0]
box_y[large_car], box_size[large_car] = 0.85, 2.0 * (0.02 + 0.08 * 0.85)
near_horizon = np.flatnonzero((d_video == 6) & (d_frame == 60) & (d_track == 1))[0]
box_y[near_horizon], box_size[near_horizon] = 0.25, 0.02 + 0.08 * 0.85
metadata.add_factors({"vehicle": vehicle, "box_y": box_y, "box_size": box_size}, level="instance")

In [6]:
boxes = ["box_size", "box_y"]
display(Outliers().evaluate(metadata, levels="instance", factors=boxes).data())

item_index,metric_name,metric_value,direction,bound,percentile,population_mean,population_std
i64,cat,f64,cat,f64,f64,f64,f64


In [7]:
by_vehicle = Outliers().evaluate(metadata, levels="instance", factors=boxes, strata=["vehicle"])
display(by_vehicle.data().select("item_index", "target_index", "metric_name", "metric_value", "bound", "population"))

item_index,target_index,metric_name,metric_value,bound,population
i64,i64,cat,f64,f64,cat
2,300,"""box_size""",0.176,0.141414,"""vehicle=car"""


In [8]:
display(by_vehicle.populations)

level,metric_name,method,population,n,judged,reason
str,cat,cat,cat,i64,bool,str
"""instance""","""box_size""",null,"""vehicle=car""",3200,true,null
"""instance""","""box_size""",null,"""vehicle=truck""",1600,true,null
"""instance""","""box_size""",null,"""vehicle=bus""",12,false,"""below 20 rows"""
"""instance""","""box_y""",null,"""vehicle=car""",3200,true,null
"""instance""","""box_y""",null,"""vehicle=truck""",1600,true,null
"""instance""","""box_y""",null,"""vehicle=bus""",12,false,"""below 20 rows"""


In [9]:
rare = Outliers().evaluate(metadata, levels="unit", factors=["weather"], strata=["item_index"])
display(rare.data().select("item_index", "target_index", "observed", "population", "metric_value", "bound"))

item_index,target_index,observed,population,metric_value,bound
i64,i64,cat,cat,f64,f64
0,80,"""weather=snow""","""item_index=0""",0.023498,0.05


In [10]:
with use_seed(0):
    given_hour = Outliers().evaluate(metadata, levels="unit", factors=["brightness"], given={"brightness": "hour"})
display(given_hour.data().select("item_index", "target_index", "metric_name", "metric_value", "bound", "method"))

item_index,target_index,metric_name,metric_value,bound,method
i64,i64,cat,f64,f64,cat
0,140,"""brightness given hour""",41.751452,3.926295,"""residual"""
1,103,"""brightness given hour""",-4.690794,-3.774356,"""residual"""


In [11]:
strict = given_hour.with_threshold({"brightness given hour": ("modzscore", 6.0)})
display(strict.data().select("item_index", "target_index", "metric_name", "metric_value", "bound"))

item_index,target_index,metric_name,metric_value,bound
i64,i64,cat,f64,f64
0,140,"""brightness given hour""",41.751452,5.265822


In [12]:
with use_seed(0):
    combined = Outliers().evaluate(
        metadata, levels="instance", factors=boxes, strata=["vehicle"], combinations=[("box_size", "box_y")]
    )
display(
    combined
    .data()
    .filter(pl.col("method") == "mahalanobis")
    .select("item_index", "target_index", "metric_name", "metric_value", "bound", "population")
)

item_index,target_index,metric_name,metric_value,bound,population
i64,i64,cat,f64,f64,cat
2,300,"""box_size, box_y""",24.232803,4.142277,"""vehicle=car"""
6,181,"""box_size, box_y""",13.262775,4.142277,"""vehicle=car"""
6,553,"""box_size, box_y""",4.34158,4.142277,"""vehicle=car"""


In [13]:
print(len(Outliers().evaluate(metadata, levels="unit", factors=["exposure"])))

0


In [14]:
along = Outliers().evaluate(metadata, levels="unit", factors=["exposure"], neighbors=["exposure", "time_s"])
display(along.data().select("item_index", "target_index", "metric_name", "metric_value", "bound", "method"))

item_index,target_index,metric_name,metric_value,bound,method
i64,i64,cat,f64,f64,cat
1,73,"""exposure along sequence""",9.310004,9.274995,"""neighbors"""
4,50,"""exposure along sequence""",9.0,11.569835,"""neighbors"""
7,8,"""exposure along sequence""",15.318609,15.261744,"""neighbors"""
7,120,"""time_s along sequence""",10804.0,39.025945,"""neighbors"""
7,120,"""time_s along sequence""",10804.0,4.033333,"""order"""


In [15]:
# TEST ASSERTION CELL ###
def in_video(index: int) -> int:
    """A detection's position within its video, which is its `target_index`."""
    return int(np.sum(d_video[:index] == d_video[index]))


horizon_box, large_box = SourceIndex(6, in_video(near_horizon)), SourceIndex(2, in_video(large_car))
assert len(Outliers().evaluate(metadata, levels="instance", factors=boxes)) == 0
assert by_vehicle.outliers == {large_box: ["box_size"]}
bus = by_vehicle.populations.filter(pl.col("population").cast(pl.Utf8) == "vehicle=bus")
assert bus["judged"].to_list() == [False, False]
assert rare.data()["observed"].cast(pl.Utf8).to_list() == ["weather=snow"]
assert rare.outliers == {SourceIndex(0, 80, "unit"): ["weather"]}
assert SourceIndex(0, 140, "unit") in given_hour.outliers
assert strict.outliers == {SourceIndex(0, 140, "unit"): ["brightness given hour"]}
pairs = combined.data().filter(pl.col("method") == "mahalanobis")
assert {(6, horizon_box.key), (2, large_box.key)} <= set(
    zip(pairs["item_index"].to_list(), pairs["target_index"].to_list(), strict=True)
)
assert horizon_box not in by_vehicle.outliers
flagged = metadata.rows_at("instance").filter((pl.col("item_index") == 6) & (pl.col("target_index") == 181))
assert flagged["target_index"].to_list() == [horizon_box.key]
assert len(Outliers().evaluate(metadata, levels="unit", factors=["exposure"])) == 0
assert {(4, 50, "neighbors"), (7, 120, "neighbors"), (7, 120, "order")} <= set(
    along.data().select("item_index", "target_index", "method").rows()
)
assert along.data().filter(pl.col("metric_name") == "time_s along sequence")["target_index"].to_list() == [120, 120]